# APs plots in the time domain

Required files:
- TTL times
- TTL labels
- APs data (can be read using TTL times)
    - Spike times
    - Spike Clusters
    - Cluster info
    - Templates   
- Sleep/Wake labels
- Video location label

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.io
import pandas as pd

import os
import re

def find_files(root_folder, pattern):
    # Compile the regex pattern for matching
    regex = re.compile(pattern)
    
    matching_files = []
    for dirpath, dirnames, filenames in os.walk(root_folder):
        for filename in filenames:
            if regex.match(filename):
                file_path = os.path.join(dirpath, filename)
                matching_files.append(file_path)
                print(f"File found: {file_path}")
    return matching_files

main_folder = r'\\132.66.34.210\Pixel1\1_auditory_neuropixels_BarakH\20240312_C9_T1_NP2_-10dB_g0'
SleepWakeLabels_fname = 'labels_All.mat'
TTL_times_fname = 'TTL_times_adj.txt'
TTL_labels_fname = 'TTL_labels.txt'
AP_spike_times_fname = r'spike_times_sec.npy'
AP_spike_clusters_fname = r'spike_clusters.npy'
AP_cluster_info_fname = r'cluster_info'
AP_templates_fname = r'templates.npy'
VID_locations_fname = r'video_loc_sync.npz'

ClusterInfo_path = find_files(main_folder, AP_cluster_info_fname)[0]
Templates_path = find_files(main_folder, AP_templates_fname)[0]
SpikeTimes_path = find_files(main_folder, AP_spike_times_fname)[0]
SpikeClusters_path = find_files(main_folder, AP_spike_clusters_fname)[0]
TTL_labels_path = find_files(main_folder, TTL_labels_fname)[0]
TTL_times_path = find_files(main_folder, TTL_times_fname)[0]
SleepWakeLabels_path = find_files(main_folder, SleepWakeLabels_fname)[0]
VID_locations_path = find_files(main_folder, VID_locations_fname)[0]

In [2]:
Templates = np.load(Templates_path)
SpikeTimes = np.squeeze(np.load(SpikeTimes_path))
SpikeClusters = np.load(SpikeClusters_path)
ClusterInfo = pd.read_csv(ClusterInfo_path, sep='\t')

TTL_times = np.loadtxt(TTL_times_path)
TTL_labels = np.loadtxt(TTL_labels_path)

SleepWakeLabels =  scipy.io.loadmat(SleepWakeLabels_path)
SleepWakeLabels = np.squeeze(SleepWakeLabels['labels'])
SleepWakeTimes = np.linspace(0, 2*60*60, len(SleepWakeLabels))  # the stop value is set by the time I used for manual scoring

VID_location_data = np.load(VID_locations_path)
VID_locations_time = VID_location_data['t_vec']
VID_locations_com = VID_location_data['location_com']
VID_locations_com = np.sqrt(np.abs(VID_locations_com[1:, 0] - VID_locations_com[:-1, 0]) ** 2 + np.abs(VID_locations_com[1:, 1] - VID_locations_com[:-1, 1]) ** 2)
VID_locations_com = VID_locations_com[0:len(VID_locations_time)]

%matplotlib inline
# %matplotlib qt
plt.figure()
plt.plot(VID_locations_time, VID_locations_com, color='blue')
plt.axhline(np.mean(VID_locations_com) + 2 * np.std(VID_locations_com), color='red', linestyle='dashed')
# plt.imshow(VID_location_data['first_frame'])

RATIO_cm_per_pixel = 0.104

Save DRC data to mat file that can be read by freq_tuning.mlx

In [3]:
stimuli_stamps = np.expand_dims(TTL_labels[(TTL_labels >= 400)  & (TTL_labels <= 600)], 1)
stimuli_times = np.expand_dims(TTL_times[(TTL_labels >= 400)  & (TTL_labels <= 600)], 1)
# good_neurons = (ClusterInfo['KSLabel'] == 'good')
good_neurons = (ClusterInfo['group'] == 'good')

clusters_list = np.expand_dims(ClusterInfo.loc[good_neurons,'cluster_id'].unique(), 1)
spike_times = np.expand_dims(SpikeTimes, 1)
spike_clusters = np.expand_dims(SpikeClusters, 1)

In [4]:
cur_SoundType_trialsTimes = stimuli_times
cur_SoundType_trialsStamps = stimuli_stamps

closest_SleepWakeTimes = []
for curTTLtime in cur_SoundType_trialsTimes:
    closest_SleepWakeTimes.append(np.argmin(np.abs(SleepWakeTimes - curTTLtime)))

cur_SoundType_SleepWakeLabels = SleepWakeLabels[closest_SleepWakeTimes]

closest_MovementTimes = []
for curTTLtime in cur_SoundType_trialsTimes:
    closest_MovementTimes.append(np.argmin(np.abs(VID_locations_time - curTTLtime)))
    
VID_locations_com_at_times = VID_locations_com[closest_MovementTimes]
th_movement = np.mean(VID_locations_com) + 2 * np.std(VID_locations_com)
cur_SoundType_MovementLabels = VID_locations_com_at_times.copy()
cur_SoundType_MovementLabels[VID_locations_com_at_times <= th_movement] = 1
cur_SoundType_MovementLabels[VID_locations_com_at_times > th_movement] = 0

All_TrialsTimes = cur_SoundType_trialsTimes
First30min_TrialsTimes = cur_SoundType_trialsTimes[cur_SoundType_trialsTimes<30*60]
Last30min_TrialsTimes = cur_SoundType_trialsTimes[3*30*60<cur_SoundType_trialsTimes]

Sleep_TrialsTimes = cur_SoundType_trialsTimes[cur_SoundType_SleepWakeLabels == 3]
Sleep_TrialsStamps = cur_SoundType_trialsStamps[cur_SoundType_SleepWakeLabels == 3]

Wake_TrialsTimes = cur_SoundType_trialsTimes[(cur_SoundType_SleepWakeLabels == 2) & (cur_SoundType_MovementLabels == 1)]  ## Quiet Wake
Wake_TrialsStamps = cur_SoundType_trialsStamps[(cur_SoundType_SleepWakeLabels == 2) & (cur_SoundType_MovementLabels == 1)]  ## Quiet Wake

In [5]:
# from scipy.io import savemat
# mdic = {"stimuli_times": stimuli_times, "stimuli_stamps": stimuli_stamps, "spike_times": spike_times, "spike_clusters": spike_clusters, "clusters_list":clusters_list}
# savemat(main_folder + "\save_tuning.mat", mdic)


In [6]:
import matlab.engine
eng = matlab.engine.start_matlab()
s = eng.genpath(r'D:\BarakH_codes\Create_PSTH_for_sounds\Base_Libraries\CurrBio')
eng.addpath(s, nargout=0)

responses_wake = []
responses_sleep = []

for cur_cluster in clusters_list:
    response_wake = eng.freq_tuning_wrapper(Wake_TrialsTimes, Wake_TrialsStamps, spike_times, spike_clusters, cur_cluster)
    response_sleep = eng.freq_tuning_wrapper(Sleep_TrialsTimes, Sleep_TrialsStamps, spike_times, spike_clusters, cur_cluster)

    responses_wake.append(response_wake)
    responses_sleep.append(response_sleep)

# Stop MATLAB engine
eng.quit()

In [43]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter, maximum_filter
import pandas as pd

def detect_peaks_2d(data, threshold=0.5, min_distance=1):
    data_max = maximum_filter(data, size=min_distance)
    peaks = (data == data_max)
    peaks[data < threshold * np.max(data)] = False
    return np.where(peaks)

def plot_spectral_analysis(responses, ClusterInfo, good_neurons, clusters_list, SIGNIFICANT=True, PLOT=False):
    clusters_channels = ClusterInfo.loc[good_neurons,'ch'].to_numpy()
    significance_per_response = np.array([sub['significantBonferroni'] for sub in responses])
    times_PsthMs_per_response = np.array([sub['timesPsthMs'] for sub in responses])
    freqs_per_response = np.array([sub['freqs'] for sub in responses])
    psthPerFreq_per_response = np.array([sub['psthPerFreq'] for sub in responses])
    
    if SIGNIFICANT:
        good_clusters_inds = np.argwhere(significance_per_response) + 0
    else:
        good_clusters_inds = np.array(list(range(0, len(clusters_list))))
    good_clusters = clusters_list[good_clusters_inds]
    good_clusters_channels = clusters_channels[good_clusters_inds]
    good_significance = significance_per_response[good_clusters_inds]
    
    num_plots = len(good_clusters_inds)
    cols = 2  # Number of columns in the subplot
    rows = (num_plots + cols - 1) // cols  # Calculate rows needed
    
    if PLOT:
        fig, axes = plt.subplots(rows, cols, figsize=(15, 5*rows + 1))
        axes = axes.flatten()
    else:
        fig, axes = None, None
    
    # Initialize lists to store data for DataFrame
    df_data = []
    
    for i, ind in enumerate(good_clusters_inds):
        # ind = ind[0]  # get the actual index from the array
        x = np.squeeze(times_PsthMs_per_response[ind])
        y = np.squeeze(freqs_per_response[ind]) / 1000
        z = np.squeeze(psthPerFreq_per_response[ind])
        z = z[:-1,:]
        
        z_smoothed = gaussian_filter(z, 3)
        if PLOT:
            ax = axes[i]
            im = ax.pcolormesh(x, y, z_smoothed, cmap='viridis', shading='auto')
            ax.set_xlabel(r'$t$ [msec]')
            ax.set_ylabel(r'$f$ [kHz]')
            ax.set_title(f"{good_clusters[i]}, {good_clusters_channels[i]}, {good_significance[i]}")
            ax.set_ylim([np.min(y), 40])
            fig.colorbar(im, ax=ax)
        
            ax.invert_yaxis()
        
        # Detect and plot peaks
        peak_y, peak_x = detect_peaks_2d(z_smoothed, threshold=0.9, min_distance=3)
        peak_times = x[peak_x]
        peak_freqs = y[peak_y]
        peak_values = z_smoothed[peak_y, peak_x]
        if PLOT:
            ax.plot(peak_times, peak_freqs, 'r.', markersize=10)
        
        # Detect and plot troughs (negative peaks)
        trough_y, trough_x = detect_peaks_2d(-z_smoothed, threshold=0.9, min_distance=3)
        trough_times = x[trough_x]
        trough_freqs = y[trough_y]
        trough_values = z_smoothed[trough_y, trough_x]
        if PLOT:
            ax.plot(trough_times, trough_freqs, 'b.', markersize=10)
        
        # Store data for DataFrame
        cluster_id = np.squeeze(good_clusters[i, :])
        channel = np.squeeze(good_clusters_channels[i])
        significance = np.squeeze(good_significance[i])
        
        for pt, pf, pv in zip(peak_times, peak_freqs, peak_values):
            df_data.append({
                'cluster_id': cluster_id,
                'channel': channel,
                'significance': significance,
                'type': 'peak',
                'time': pt,
                'frequency': pf,
                'value': pv
            })
        
        for tt, tf, tv in zip(trough_times, trough_freqs, trough_values):
            df_data.append({
                'cluster_id': cluster_id,
                'channel': channel,
                'significance': significance,
                'type': 'trough',
                'time': tt,
                'frequency': tf,
                'value': tv
            })
    
    if PLOT:
        # Remove any empty subplots
        for j in range(i+1, len(axes)):
            fig.delaxes(axes[j])
    
    if PLOT:
        # Add suptitle before tight_layout
        fig.suptitle(f'{len(good_clusters_inds)} good responses', fontsize=16, y=0.995)
        plt.tight_layout()
        plt.subplots_adjust(top=0.96)  # Adjust top margin for suptitle
    
    # Create DataFrame
    df = pd.DataFrame(df_data)
    
    return fig, axes, df

fig_sleep, axes_sleep, peak_trough_df_sleep = plot_spectral_analysis(responses_sleep, ClusterInfo, good_neurons, clusters_list, SIGNIFICANT=True, PLOT=True)
fig_wake, axes_wake, peak_trough_df_wake = plot_spectral_analysis(responses_wake, ClusterInfo, good_neurons, clusters_list, SIGNIFICANT=True, PLOT=True)

In [40]:
display(peak_trough_df_sleep)
display(peak_trough_df_wake)

In [41]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_channel_response_comparison(df, type_plot):
    # Set up the plot style
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))
    sns.set_style("whitegrid")
    
    # Color map for peak and trough
    color_map = {'peak': 'red', 'trough': 'blue'}
    
    # Function to plot scatter for each significance group
    def plot_scatter(ax, data, x):
        for _, row in data.iterrows():
            facecolor = color_map[row['type']] if row['significance'] else 'none'
            edgecolor = color_map[row['type']]
            ax.scatter(row[x], row['channel'], 
                       facecolor=facecolor, edgecolor=edgecolor, 
                       s=np.abs(row['value']) * 50,  # Adjust size scaling as needed
                       alpha=0.7, marker='o')
    
    # Plot 1: Channel vs Frequency
    plot_scatter(ax1, df, 'frequency')
    
    ax1.set_title('Channel vs Frequency', fontsize=16)
    ax1.set_xlabel('Frequency (kHz)', fontsize=12)
    ax1.set_ylabel('Channel', fontsize=12)
    ax1.set_ylim(1, 140)
    ax1.set_xlim(0, 40)
    
    # Plot 2: Channel vs Time
    plot_scatter(ax2, df, 'time')
    
    ax2.set_title('Channel vs Time', fontsize=16)
    ax2.set_xlabel('Time (ms)', fontsize=12)
    ax2.set_ylabel('Channel', fontsize=12)
    ax2.set_ylim(1, 140)
    ax2.set_xlim(-50, 100)
    
    # Create custom legend
    legend_elements = [
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='red', markersize=10, label='Significant Peak'),
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='blue', markersize=10, label='Significant Trough'),
        plt.Line2D([0], [0], marker='o', color='red', markerfacecolor='none', markersize=10, label='Non-significant Peak'),
        plt.Line2D([0], [0], marker='o', color='blue', markerfacecolor='none', markersize=10, label='Non-significant Trough')
    ]
    
    # ax1.legend(handles=legend_elements, title='Response Type', title_fontsize='12', fontsize='10', loc='upper right')
    # ax2.legend(handles=legend_elements, title='Response Type', title_fontsize='12', fontsize='10', loc='upper right')
    
    # Adjust layout and add main title
    plt.tight_layout()
    fig.suptitle(f'Channel Response Comparison {type_plot}', fontsize=20, y=1.05)
    
    return fig

fig_sleep = plot_channel_response_comparison(peak_trough_df_sleep, "Sleep")
fig_wake = plot_channel_response_comparison(peak_trough_df_wake, "Wake")

plt.show()